In [ ]:
from sqlalchemy.testing.suite.test_reflection import metadata

In [ ]:
# !uv pip install -U langchain langchain-openai qdrant-client langchain-qdrant langgraph langchain-upstage langchain-community langchain-teddynote pip-chill python-dotenv mysql-connector-python

In [ ]:
from qdrant_client.http.models import VectorParams, Distance
from langchain_qdrant import QdrantVectorStore
from langchain_upstage import UpstageEmbeddings
from langchain_core.documents import Document
from qdrant_client import QdrantClient, models
import mysql.connector
import os
from dotenv import find_dotenv, load_dotenv

# 1. 환경 변수 로드 (UPSTAGE_API_KEY 등)
load_dotenv(find_dotenv())

# 2. DB 설정
db_config = {
    'host': os.getenv('DB_HOST'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME')
}

# 3. 배치 설정
BATCH_SIZE = 400
collection_name = "attractions_overview"  # Qdrant에 저장될 컬렉션 이름
qdrant_url = os.getenv('QDRANT_URL')  # Qdrant 서버 주소

# 4. 모델 및 클라이언트 초기화
embedding_model = UpstageEmbeddings(model="solar-embedding-1-large-passage")
dimension = len(embedding_model.embed_query("check"))  # 4096
# Qdrant 클라이언트 (데이터 저장을 위해 필요)
client = QdrantClient(url=qdrant_url) #메모리에 저장을 원할 시 - :memory:
if not client.collection_exists(
        collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
    )

# 5. LangChain Qdrant 저장소 연결
# 컬렉션이 없으면 LangChain이 자동으로 생성하려 시도하지만,
# 운영 환경에서는 미리 생성해두는 것이 좋습니다.
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embedding_model,
)

In [2]:

try:
    connection = mysql.connector.connect(**db_config)
    cursor = connection.cursor(dictionary=True)

    offset = 0
    total_inserted = 0

    print("--- 배치 처리 시작 ---")

    for _ in range(1):  #while True:
        # 6. SQL 쿼리 작성 (위치 정보 mapy, mapx 필수 포함)
        # mapy: 위도(latitude), mapx: 경도(longitude)라고 가정 (공공데이터 포맷)
        # overview가 비어있지 않은 데이터만 조회
        query = f"""
            SELECT *
            FROM attractions
            WHERE overview IS NOT NULL AND overview != ''
            LIMIT {BATCH_SIZE} OFFSET {offset}
        """

        cursor.execute(query)
        rows = cursor.fetchall()

        # 가져올 데이터가 없으면 루프 종료
        if not rows:
            print("--- 모든 데이터 처리 완료 ---")
            break

        documents = [Document(
            page_content=row["overview"],
            metadata={
                "location": {
                    "lat": float(row["latitude"]),
                    "lon": float(row["longitude"])
                },
                **{k: v for k, v in row.items() if k not in ["overview", "latitude", "longitude"]}
            }
        ) for row in rows if float(row["latitude"]) != 0 and float(row["longitude"]) != 0]
        # 8. 벡터 스토어에 저장 (배치 단위)
        if documents:
            vector_store.add_documents(documents)
            total_inserted += len(documents)
            print(f"Current Batch: {len(documents)}건 저장 (누적: {total_inserted}건) | Offset: {offset}")

        # 다음 배치를 위해 offset 증가
        offset += BATCH_SIZE

except mysql.connector.Error as err:
    print(f"Error: {err}")
finally:
    if 'connection' in locals() and connection.is_connected():
        cursor.close()
        connection.close()
        print("MySQL 연결 종료")

--- 배치 처리 시작 ---
Current Batch: 400건 저장 (누적: 400건) | Offset: 0
MySQL 연결 종료


In [3]:
print(documents)

[Document(metadata={'location': {'lat': 37.5820858828, 'lon': 126.9846616856}, 'no': 56644, 'content_id': 2733967, 'title': '가회동성당', 'content_type_id': 12, 'area_code': 1, 'si_gun_gu_code': 23, 'first_image1': 'http://tong.visitkorea.or.kr/cms/resource/09/3303909_image2_1.jpg', 'first_image2': 'http://tong.visitkorea.or.kr/cms/resource/09/3303909_image3_1.jpg', 'map_level': 6, 'tel': '', 'addr1': '서울특별시 종로구 북촌로 57 (가회동)', 'addr2': '', 'homepage': '<a href="https://gahoe.or.kr" target="_blank" title="새창 : 가회동성당 사이트로 이동">https://gahoe.or.kr</a>', 'area_code_id': None, 'content_type': None}, page_content='가회동성당이 위치한 북촌 일대는 최초의 선교사 주문모(周文謨, 야고보) 신부가 조선에 밀입국하여 1795년 4월 5일 부활 대축일에 최인길(崔仁吉, 마티아)의 집에서 조선 땅에서의 ‘첫 미사’를 집전한 지역이다. 본당 관할구역은 주문모 신부가 강완숙(姜完淑, 골롬바)의 집에 숨어 지내면서 사목활동을 펼쳤던 지역으로서 한국 교회사에서 매우 중요한 의미가 있다.정식으로 본당이 된 것은 1949년이고, 이후 1954년에 성전이 완공되었다. 하지만 성전이 낡아 2011년부터 옛 성전을 허물고 현재의 새 성전을 짓게 되었다. 2013년 11월 21일 준공되었고, 준공 3일 후인 24일(그리스도 왕 대축일)에 입주하여 입주 미사를 봉헌하였다. 현재의 동서양 건축양식이 어우러진 새 성전은 과거의 역사를

In [14]:
# 5. 실전 검색 (위치 필터 + 앙상블)
user_query = ("근처에 기분 전환 겸 쇼핑하기 좋은 곳 있어? 가급적이면 카페도 있었으면 좋겠고.")
user_loc = {"lat": 37.5660, "lon": 126.9784}  # 서울 시청

# 5-1. 쿼리 벡터화
filter = models.Filter(
    must=[
        models.FieldCondition(
            key="metadata.location",
            geo_radius=models.GeoRadius(
                center=models.GeoPoint(lat=user_loc["lat"], lon=user_loc["lon"]),
                radius=1000
            )
        )
    ]
)
# 5-2. 검색 실행
results = vector_store.similarity_search_with_score(
    user_query,
    k=10,
    filter=filter,
    score_threshold=0.22
)

print(f"--- 검색어: '{user_query}' (1km 이내) ---")
for doc,score in results:
    print(f"[{score:.4f}] {doc.metadata['title']} :{doc.page_content}")

--- 검색어: '근처에 기분 전환 겸 쇼핑하기 좋은 곳 있어? 가급적이면 카페도 있었으면 좋겠고.' (1km 이내) ---
[0.2930] 명동 :명동은 거대 쇼핑도시를 연상케하는 공간이다. 일반적으로 명동 거리는 지하철 4호선 명동역에서 을지로, 롯데백화점으로 이어지는 약 1km 정도의 거리를 말한다. 이곳에는 각종 브랜드매장, 백화점,보세가게 등이 밀집되어 있다. 유행의 메카라는 표현이 어울릴 정도로 의류,신발,액세서리 등의 다양한 제품을 구입할 수 있다. 남대문이나 동대문보다는 질이 좋은 브랜드가 많이 모여 있는 것이 특징이다.우선 백화점으로는 가까이에 롯데백화점, 신세계백화점이 있으며, 명동거리에는 눈스퀘어(Noon Square), 명동밀레오레, 엠플라자(M Plaza)와 같은 쇼핑몰이 있다. 각종 브랜드숍은 중앙거리를 비롯해 사이드 골목에 밀집되어 있다.명동에는 쇼핑과 함께 먹을거리, 즐길거리가 많다.먹을거리로는 패밀리레스토랑, 패스트푸드점은 물론 한식, 양식, 일식으로 다양하다. 이중에서 명동 돈까스와 칼국수(명동교자)는 유명하므로 한번 먹어보는 것이 좋다. 그 외에도 명동에는 헤어샵, 은행, 극장 등 많은 편의시설이 있다.* 명동 주요 관광지- 명동성당 : 한국 천주교 서울대교구 본당 건물인 명동성당은 우리나라 최초의 본당이며 한국 천주교회의 상징으로, 고종 29년(1892)에 착공, 광무 2년(1898)에 준공된 순수한 고딕양식의 건물이다. 명동성당은 우리나라 기독교 역사뿐 아니라 정치, 사회, 문화 전반에 걸쳐 큰 영향을 미친 터전이다.- 명동예술극장 : 명동 옛 국립극장을 복원하여 2009년 6월 개관한 명동예술극장은 완성도 높은 연극작품을 만날 수 있는 연극예술전문 공연장이다. 1934년 바로크 양식으로 건축된 극장외형과 현대적인 내부공연 시설이 잘 조화되는 곳이다.- 명동 재미로 : 명동과 남산의 연결지점이지만 명동의 화려함에 비해 특색 없이 밋밋하고 가파르기만 했던 오르막길이 2013년 12월, 만화의 거리 '재미로'라는 이름으로 새롭